### Train Pipeline
MVP training notebook for `cvm_churn-from-dac_binary-class_churn-dac`.

The notebook mirrors the deployment train logic: train/holdout are loaded from the preprocess S3 prefix, CV is calculated only on train, final validation is calculated on holdout, and optional OOT validation can be run for the next month dataset.

### 1. Imports / Env / Flags

In [ ]:
%load_ext autoreload
%load_ext dotenv
%dotenv
%autoreload 2

from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List

import logging
import os
import time

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from IPython.display import display
from mlflow.models.signature import infer_signature
from sklearn.metrics import (
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    fbeta_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import calibration_curve

import cvm_ml_metrics.classification as mc
from cvm_model.io import State
import cvm_model.utils as utils
from cvm_model.parameters import (
    RANDOM_STATE,
    artifacts_dir,
    features,
    input_suffix,
    jira,
    model_type,
    project_name,
    score,
    target,
    template,
    threshold,
)

logging.basicConfig(level=logging.INFO)
pd.set_option('display.max_columns', None)

RUN_OPTUNA = False
RUN_MLFLOW = True
RUN_OOT = False
RUN_SHAP = True

N_FOLDS = 5
OPTUNA_TRIALS = 30
SHAP_SAMPLE_SIZE = 50_000

# This timestamp must match the preprocess run timestamp.
event_timestamp = datetime(2026, 2, 1)

# Optional OOT path. It should point to a parquet dataset with the same features and target.
OOT_PATH = None

artifacts_path = Path(artifacts_dir)
plots_path = artifacts_path / 'plots'
artifacts_path.mkdir(parents=True, exist_ok=True)
plots_path.mkdir(parents=True, exist_ok=True)

print('features:', len(features))
print('target:', target)
print('threshold from parameters.py:', threshold)

### 2. State and Dataset Loading

In [ ]:
state = State.from_env()
spark = state.spark.session

# Same input path logic as in train.py and preprocess.py.
input_prefix = state.settings.preprocess_prefix(event_timestamp) / input_suffix
input_bucket = input_prefix.split('//')[1].split('/')[0]
input_prefix = '/'.join(input_prefix.split('//')[1].split('/')[1:])
input_path = template.format(bucket=input_bucket, prefix=input_prefix)

train_path = f'{input_path}/train'
holdout_path = f'{input_path}/holdout'

print('train_path:', train_path)
print('holdout_path:', holdout_path)

df = utils.load_dataset(spark, train_path)
holdout_df = utils.load_dataset(spark, holdout_path)

print('Train shape:', df.shape)
print('Holdout shape:', holdout_df.shape)

display(df.head())

### 3. Dataset Checks

In [ ]:
missing_features = sorted(set(features) - set(df.columns))
missing_holdout_features = sorted(set(features) - set(holdout_df.columns))
assert not missing_features, f'Missing train features: {missing_features}'
assert not missing_holdout_features, f'Missing holdout features: {missing_holdout_features}'
assert target in df.columns, f'{target} is missing in train dataset'
assert target in holdout_df.columns, f'{target} is missing in holdout dataset'
assert df['contact_id'].is_unique, 'Train contact_id is not unique'
assert holdout_df['contact_id'].is_unique, 'Holdout contact_id is not unique'
assert not (set(df['contact_id']) & set(holdout_df['contact_id'])), 'Train and holdout contact_id overlap'

print('Train target rate:', df[target].mean())
print('Holdout target rate:', holdout_df[target].mean())

if 'dac_segment_12m' in df.columns:
    print('\nTrain segments:')
    display(df.groupby('dac_segment_12m')[target].agg(['count', 'mean']).sort_values('mean', ascending=False))

if 'dac_segment_12m' in holdout_df.columns:
    print('\nHoldout segments:')
    display(holdout_df.groupby('dac_segment_12m')[target].agg(['count', 'mean']).sort_values('mean', ascending=False))

### 4. Metrics Helpers

In [ ]:
metrics_lib: Dict[str, Any] = {
    'log_loss': log_loss,
    'sensitivity': lambda x, y: mc.sensitivity_specificity(x, y)[0],
    'specificity': lambda x, y: mc.sensitivity_specificity(x, y)[1],
    'balanced_accuracy_lib': mc.balanced_accuracy,
    'youden_j': mc.youden_j,
    'matthews_corrcoef': matthews_corrcoef,
    'cohen_kappa_score': mc.cohen_kappa_score,
}

metrics_lib_proba: Dict[str, Any] = {
    'precision_recall_auc': mc.precision_recall_auc,
    'brier_score_loss': brier_score_loss,
    'markedness': mc.markedness,
    'lift': mc.lift,
    'gini': mc.gini,
    'ks_stat_bin_class': mc.ks_stat_bin_class,
    'ece': lambda x, y: mc.ece_mce_fast(x, y)[0],
    'mce': lambda x, y: mc.ece_mce_fast(x, y)[1],
}


def evaluate_metrics(data: pd.DataFrame, threshold_value: float, prefix: str) -> Dict[str, float]:
    y_true = data[target].to_numpy()
    y_pred_proba = data[score].to_numpy()
    y_pred = (y_pred_proba >= threshold_value).astype(int)

    metrics = {
        f'{prefix} precision': precision_score(y_true, y_pred, zero_division=0),
        f'{prefix} recall': recall_score(y_true, y_pred, zero_division=0),
        f'{prefix} f1': f1_score(y_true, y_pred, zero_division=0),
        f'{prefix} f05': fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
        f'{prefix} f2': fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        f'{prefix} balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
    }

    for metric_name, method in metrics_lib_proba.items():
        metrics[f'{prefix} {metric_name}'] = method(y_true, y_pred_proba)

    for metric_name, method in metrics_lib.items():
        metric_input = y_pred_proba if metric_name == 'log_loss' else y_pred
        metrics[f'{prefix} {metric_name}'] = method(y_true, metric_input)

    return metrics


def decile_report(data: pd.DataFrame, n_bins: int = 10) -> pd.DataFrame:
    result = data[[target, score]].copy()
    result['score_decile'] = pd.qcut(result[score].rank(method='first'), n_bins, labels=False) + 1
    result = result.groupby('score_decile').agg(
        rows=(target, 'size'),
        target_rate=(target, 'mean'),
        score_min=(score, 'min'),
        score_max=(score, 'max'),
        score_mean=(score, 'mean'),
    ).sort_index(ascending=False)
    return result


def brier_decomposition(y_true, y_prob, n_bins: int = 10) -> Dict[str, float]:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    indices = np.digitize(y_prob, bins) - 1
    indices = np.clip(indices, 0, n_bins - 1)
    bin_totals = np.bincount(indices, minlength=n_bins)
    bin_sums = np.bincount(indices, weights=y_prob, minlength=n_bins)
    bin_true_sums = np.bincount(indices, weights=y_true, minlength=n_bins)
    mask = bin_totals > 0
    bin_means = bin_sums[mask] / bin_totals[mask]
    bin_true_means = bin_true_sums[mask] / bin_totals[mask]
    reliability = np.sum(bin_totals[mask] * (bin_means - bin_true_means) ** 2) / len(y_true)
    uncertainty = np.mean(y_true) * (1 - np.mean(y_true))
    resolution = np.sum(bin_totals[mask] * (bin_true_means - np.mean(y_true)) ** 2) / len(y_true)
    return {
        'brier_reliability': reliability,
        'brier_resolution': resolution,
        'brier_uncertainty': uncertainty,
    }

### 5. Optuna Tuning Optional

In [ ]:
base_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'PRAUC',
    'random_seed': RANDOM_STATE,
    'early_stopping_rounds': 50,
    'thread_count': -1,
    'verbose': False,
}

best_params = {
    'iterations': 1200,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 6,
    'random_strength': 1,
    'bagging_temperature': 1,
}

if RUN_OPTUNA:
    import optuna

    def objective(trial):
        params = {
            **base_params,
            'iterations': trial.suggest_int('iterations', 500, 2000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 20, log=True),
            'random_strength': trial.suggest_float('random_strength', 0, 5),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 5),
        }

        cv_scores = []
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        for train_idx, val_idx in skf.split(df[features], df[target]):
            train_part = df.iloc[train_idx]
            val_part = df.iloc[val_idx]
            counter = Counter(train_part[target].to_numpy())
            minority_to_majority_ratio = counter[1] / counter[0] if counter[0] else 1.0
            model = CatBoostClassifier(**params, class_weights=[minority_to_majority_ratio, 1.0])
            model.fit(
                train_part[features],
                train_part[target],
                eval_set=(val_part[features], val_part[target]),
                use_best_model=True,
            )
            pred = model.predict_proba(val_part[features])[:, 1]
            cv_scores.append(mc.precision_recall_auc(val_part[target], pred))
        return float(np.mean(cv_scores))

    study = optuna.create_study(direction='maximize', study_name='catboost_pr_auc')
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    best_params = study.best_params
    print('Best Optuna score:', study.best_value)
    print('Best params:', best_params)
else:
    print('Optuna is disabled. Using default CatBoost params:')
    print(best_params)

### 6. StratifiedKFold Cross-Validation / OOF

In [ ]:
oof = df[['contact_id', target]].copy()
oof[score] = np.nan
fold_metrics = []
best_iterations = []
models_cv = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(skf.split(df[features], df[target])):
    print(f'Fold {fold}')
    train_part = df.iloc[train_idx]
    val_part = df.iloc[val_idx]

    counter = Counter(train_part[target].to_numpy())
    minority_to_majority_ratio = counter[1] / counter[0] if counter[0] else 1.0

    model = CatBoostClassifier(
        **base_params,
        **best_params,
        class_weights=[minority_to_majority_ratio, 1.0],
    )
    model.fit(
        train_part[features],
        train_part[target],
        eval_set=(val_part[features], val_part[target]),
        use_best_model=True,
    )

    val_score = model.predict_proba(val_part[features])[:, 1]
    oof.loc[oof.index[val_idx], score] = val_score
    fold_df = val_part[[target]].copy()
    fold_df[score] = val_score
    fold_metrics.append(evaluate_metrics(fold_df, threshold, f'fold_{fold}'))
    best_iterations.append(model.get_best_iteration())
    models_cv.append(model)

assert oof[score].notna().all(), 'OOF scores contain NaN'

cv_metrics = evaluate_metrics(oof, threshold, 'oof')
cv_metrics.update({f'oof {k}': v for k, v in brier_decomposition(oof[target].to_numpy(), oof[score].to_numpy()).items()})

print('Mean best_iteration:', np.mean(best_iterations))
print('OOF metrics with default threshold:')
display(pd.Series(cv_metrics).to_frame('value'))

### 7. Threshold Calibration

In [ ]:
threshold_grid = np.arange(0.01, 0.81, 0.01)
threshold_report = []

for thr in threshold_grid:
    y_pred = (oof[score] >= thr).astype(int)
    threshold_report.append({
        'threshold': thr,
        'predicted_positive_share': y_pred.mean(),
        'precision': precision_score(oof[target], y_pred, zero_division=0),
        'recall': recall_score(oof[target], y_pred, zero_division=0),
        'f1': f1_score(oof[target], y_pred, zero_division=0),
        'f05': fbeta_score(oof[target], y_pred, beta=0.5, zero_division=0),
        'f2': fbeta_score(oof[target], y_pred, beta=2, zero_division=0),
    })

threshold_report = pd.DataFrame(threshold_report)
best_threshold_f1 = float(threshold_report.loc[threshold_report['f1'].idxmax(), 'threshold'])
best_threshold_f2 = float(threshold_report.loc[threshold_report['f2'].idxmax(), 'threshold'])
selected_threshold = threshold

print('Default threshold from parameters.py:', threshold)
print('Best threshold by F1:', best_threshold_f1)
print('Best threshold by F2:', best_threshold_f2)
print('Selected threshold for MVP run:', selected_threshold)

display(threshold_report.sort_values('f1', ascending=False).head(10))

plt.figure(figsize=(10, 5))
plt.plot(threshold_report['threshold'], threshold_report['precision'], label='precision')
plt.plot(threshold_report['threshold'], threshold_report['recall'], label='recall')
plt.plot(threshold_report['threshold'], threshold_report['f1'], label='f1')
plt.plot(threshold_report['threshold'], threshold_report['f2'], label='f2')
plt.axvline(threshold, color='black', linestyle='--', label=f'default={threshold}')
plt.axvline(best_threshold_f1, color='red', linestyle=':', label=f'best_f1={best_threshold_f1:.2f}')
plt.title('Threshold Calibration on OOF')
plt.xlabel('threshold')
plt.ylabel('metric')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(plots_path / 'threshold_calibration_oof.png', bbox_inches='tight')
plt.show()

### 8. Final Model Train

In [ ]:
counter = Counter(df[target].to_numpy())
minority_to_majority_ratio = counter[1] / counter[0] if counter[0] else 1.0

final_model = CatBoostClassifier(
    **base_params,
    **best_params,
    class_weights=[minority_to_majority_ratio, 1.0],
)

final_model.fit(
    df[features],
    df[target],
    eval_set=(holdout_df[features], holdout_df[target]),
    use_best_model=True,
    plot=True,
    plot_file=str(artifacts_path / 'training_plot.html'),
)

print('Final model best_iteration:', final_model.get_best_iteration())

### 9. Holdout Evaluation

In [ ]:
df_scored = df.copy()
holdout_scored = holdout_df.copy()

df_scored[score] = final_model.predict_proba(df_scored[features])[:, 1]
holdout_scored[score] = final_model.predict_proba(holdout_scored[features])[:, 1]

train_metrics = evaluate_metrics(df_scored, selected_threshold, 'train')
holdout_metrics = evaluate_metrics(holdout_scored, selected_threshold, 'holdout')
holdout_metrics_default = evaluate_metrics(holdout_scored, threshold, 'holdout_default_threshold')
holdout_metrics.update({f'holdout {k}': v for k, v in brier_decomposition(holdout_scored[target], holdout_scored[score]).items()})

print('Holdout metrics:')
display(pd.Series(holdout_metrics).to_frame('value'))

print('Holdout decile report:')
holdout_deciles = decile_report(holdout_scored)
display(holdout_deciles)

if 'dac_segment_12m' in holdout_scored.columns:
    segment_report = holdout_scored.groupby('dac_segment_12m').apply(
        lambda x: pd.Series({
            'rows': len(x),
            'target_rate': x[target].mean(),
            'pr_auc': mc.precision_recall_auc(x[target], x[score]) if x[target].nunique() > 1 else np.nan,
            'roc_auc': roc_auc_score(x[target], x[score]) if x[target].nunique() > 1 else np.nan,
            'score_mean': x[score].mean(),
        })
    ).sort_values('target_rate', ascending=False)
    display(segment_report)

### 10. Additional Graphs

In [ ]:
y_true = holdout_scored[target].to_numpy()
y_score = holdout_scored[score].to_numpy()
y_pred = (y_score >= selected_threshold).astype(int)

# PR curve
precision_arr, recall_arr, _ = precision_recall_curve(y_true, y_score)
plt.figure(figsize=(7, 5))
plt.plot(recall_arr, precision_arr, label=f'PR-AUC={mc.precision_recall_auc(y_true, y_score):.4f}')
plt.axhline(y_true.mean(), color='black', linestyle='--', label=f'base={y_true.mean():.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Holdout Precision-Recall Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(plots_path / 'holdout_pr_curve.png', bbox_inches='tight')
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_score)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'ROC-AUC={roc_auc_score(y_true, y_score):.4f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='black')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Holdout ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(plots_path / 'holdout_roc_curve.png', bbox_inches='tight')
plt.show()

# Calibration curve
prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10, strategy='quantile')
plt.figure(figsize=(7, 5))
plt.plot(prob_pred, prob_true, marker='o', label='model')
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='perfect')
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed target rate')
plt.title('Holdout Calibration Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(plots_path / 'holdout_calibration_curve.png', bbox_inches='tight')
plt.show()

# Score distribution
plt.figure(figsize=(8, 5))
sns.histplot(holdout_scored.loc[holdout_scored[target] == 0, score], label='not churn', stat='density', kde=True)
sns.histplot(holdout_scored.loc[holdout_scored[target] == 1, score], label='churn', stat='density', kde=True, alpha=0.35)
plt.axvline(selected_threshold, color='black', linestyle='--', label=f'threshold={selected_threshold}')
plt.title('Holdout Score Distribution by Target')
plt.legend()
plt.savefig(plots_path / 'holdout_score_distribution.png', bbox_inches='tight')
plt.show()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['not churn', 'churn'])
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Holdout Confusion Matrix')
plt.savefig(plots_path / 'holdout_confusion_matrix.png', bbox_inches='tight')
plt.show()

# Feature importance
i = pd.DataFrame({
    'feature': features,
    'importance': final_model.get_feature_importance(),
}).sort_values('importance', ascending=False)

display(i.head(30))
plt.figure(figsize=(8, 10))
sns.barplot(data=i.head(30), x='importance', y='feature')
plt.title('Top 30 CatBoost Feature Importance')
plt.savefig(plots_path / 'feature_importance_top30.png', bbox_inches='tight')
plt.show()

# SHAP
if RUN_SHAP:
    import shap
    sample = holdout_scored[features].sample(min(SHAP_SAMPLE_SIZE, len(holdout_scored)), random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(sample)
    shap.summary_plot(shap_values, sample, show=False, max_display=30)
    plt.title('SHAP Summary Plot')
    plt.savefig(plots_path / 'shap_summary.png', bbox_inches='tight')
    plt.show()

### 11. OOT Evaluation Optional

In [ ]:
oot_metrics = {}
oot_scored = None

if RUN_OOT:
    assert OOT_PATH is not None, 'Set OOT_PATH before running OOT evaluation'
    print('Loading OOT:', OOT_PATH)

    if str(OOT_PATH).startswith('s3'):
        oot_df = spark.read.parquet(OOT_PATH).toPandas()
    else:
        oot_df = pd.read_parquet(OOT_PATH)

    missing_oot_features = sorted(set(features) - set(oot_df.columns))
    assert not missing_oot_features, f'Missing OOT features: {missing_oot_features}'

    oot_scored = oot_df.copy()
    oot_scored[score] = final_model.predict_proba(oot_scored[features])[:, 1]

    if target in oot_scored.columns:
        oot_metrics = evaluate_metrics(oot_scored, selected_threshold, 'oot')
        oot_metrics.update({f'oot {k}': v for k, v in brier_decomposition(oot_scored[target], oot_scored[score]).items()})
        display(pd.Series(oot_metrics).to_frame('value'))
        display(decile_report(oot_scored))

        if 'dac_segment_12m' in oot_scored.columns:
            display(oot_scored.groupby('dac_segment_12m').apply(
                lambda x: pd.Series({
                    'rows': len(x),
                    'target_rate': x[target].mean(),
                    'pr_auc': mc.precision_recall_auc(x[target], x[score]) if x[target].nunique() > 1 else np.nan,
                    'roc_auc': roc_auc_score(x[target], x[score]) if x[target].nunique() > 1 else np.nan,
                    'score_mean': x[score].mean(),
                })
            ).sort_values('target_rate', ascending=False))
    else:
        print('OOT target is missing: only scores were calculated')
        display(oot_scored[[score]].describe())
else:
    print('OOT evaluation is disabled')

### 12. MLflow Logging

In [ ]:
model_name = f'{project_name}_{model_type}'
experiment_name = f'{model_name}_{jira}_train'
run_name = f'train_{event_timestamp.date().isoformat()}_mvp'

all_metrics = {}
all_metrics.update({k.replace(' ', '_'): float(v) for k, v in cv_metrics.items()})
all_metrics.update({k.replace(' ', '_'): float(v) for k, v in train_metrics.items()})
all_metrics.update({k.replace(' ', '_'): float(v) for k, v in holdout_metrics.items()})
all_metrics.update({k.replace(' ', '_'): float(v) for k, v in oot_metrics.items()})

params = {
    'event_timestamp': event_timestamp.date().isoformat(),
    'threshold_default': threshold,
    'threshold_selected': selected_threshold,
    'threshold_best_f1_oof': best_threshold_f1,
    'threshold_best_f2_oof': best_threshold_f2,
    'n_folds': N_FOLDS,
    'run_optuna': RUN_OPTUNA,
    'train_rows': len(df),
    'holdout_rows': len(holdout_df),
    'train_target_rate': df[target].mean(),
    'holdout_target_rate': holdout_df[target].mean(),
    'best_iteration': final_model.get_best_iteration(),
}
params.update({f'catboost_{k}': v for k, v in best_params.items()})

features_file = artifacts_path / 'features.txt'
features_file.write_text('\n'.join(features), encoding='utf-8')
threshold_report_file = artifacts_path / 'threshold_report.csv'
threshold_report.to_csv(threshold_report_file, index=False)
holdout_decile_file = artifacts_path / 'holdout_decile_report.csv'
holdout_deciles.to_csv(holdout_decile_file)
feature_importance_file = artifacts_path / 'feature_importance.csv'
i.to_csv(feature_importance_file, index=False)

if RUN_MLFLOW:
    mlflow.set_experiment(experiment_name)
    signature = infer_signature(df.head(1)[features], df_scored.head(1)[score])

    with mlflow.start_run(run_name=run_name, description=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(all_metrics)

        for path in artifacts_path.rglob('*'):
            if path.is_file() and '.ipynb_checkpoints' not in str(path):
                mlflow.log_artifact(str(path))

        mlflow.sklearn.log_model(
            final_model,
            name=model_name,
            signature=signature,
            registered_model_name=model_name,
            input_example=df.head(3)[features],
        )

    print('Logged to MLflow experiment:', experiment_name)
else:
    print('MLflow logging is disabled')

### 13. Summary

In [ ]:
summary = pd.DataFrame([
    {'dataset': 'OOF train', 'rows': len(oof), 'target_rate': oof[target].mean(), 'pr_auc': mc.precision_recall_auc(oof[target], oof[score]), 'roc_auc': roc_auc_score(oof[target], oof[score])},
    {'dataset': 'Holdout', 'rows': len(holdout_scored), 'target_rate': holdout_scored[target].mean(), 'pr_auc': mc.precision_recall_auc(holdout_scored[target], holdout_scored[score]), 'roc_auc': roc_auc_score(holdout_scored[target], holdout_scored[score])},
])

if oot_scored is not None and target in oot_scored.columns:
    summary = pd.concat([
        summary,
        pd.DataFrame([{'dataset': 'OOT', 'rows': len(oot_scored), 'target_rate': oot_scored[target].mean(), 'pr_auc': mc.precision_recall_auc(oot_scored[target], oot_scored[score]), 'roc_auc': roc_auc_score(oot_scored[target], oot_scored[score])}]),
    ], ignore_index=True)

display(summary)